In [8]:
from hydra import compose, initialize
from omegaconf import OmegaConf

with initialize(version_base=None, config_path="/enreg/config/", job_name="test_app"):
    cfg = compose(config_name="benchmarking")

In [ ]:
from enreg.tools.metrics import decay_mode_evaluator as dme
from enreg.tools.metrics import regression_evaluator as re
from enreg.tools.metrics import tagger_evaluator as te
from enreg.tools import general as g
import os
import awkward as ak

In [ ]:
cfg.comparison_samples = ['zz_test']
algorithms = ["ParticleTransformer"]
tasks = ["jet_regression"]
signal_samples = ['zz_test']

In [ ]:
data = {sample: g.load_all_data(os.path.join(cfg.NTUPLE_BASE_DIR, sample + ".parquet")) for sample in cfg.comparison_samples}

In [ ]:
task = "jet_regression"
evaluators = []
for algorithm in algorithms:
    base_path = os.path.join("/home/norman/ml-tau/ml-tau-en-reg/training-outputs/260209_jet_reg", task, algorithm)
    for signal_sample in signal_samples:
        sig_info_data = data[signal_sample]
        sig_data = g.load_all_data(os.path.join(base_path, signal_sample + ".parquet"))

        evaluator = re.RegressionEvaluator(sig_data.jet_regression.pred, sig_data.jet_regression.target, cfg.metrics.regression, signal_sample.split("_")[0], algorithm)
        evaluators.append(evaluator)
output_dir = "output_plots_regression"
rme = re.RegressionMultiEvaluator(output_dir, cfg.metrics.regression, signal_sample.split("_")[0])
rme.combine_results(evaluators)
rme.save()

In [ ]:
print(cfg.keys())

In [ ]:
print(cfg.metrics.keys())
